# KNN Coffee Bean Predictor
Predicts a coffee bean (`name`) from roast level, aroma, structure, body, flavour, and aftertaste.

In [17]:
import pandas as pd
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import OrdinalEncoder, StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
import warnings
warnings.filterwarnings('ignore')

In [18]:
df = pd.read_csv('coffee_data.csv')
df['agtron_whole'] = df['agtron'].str.extract(r'^(\d+)').astype(float)
print(f'Shape: {df.shape}')
print(f'Roast levels: {df["roast_level"].unique()}')
print('Missing values:')
print(df.isnull().sum())
df.head()

Shape: (9000, 14)
Roast levels: ['Medium-Dark' 'Medium' 'Medium-Light' 'Light' 'Dark' 'Very Dark' nan]
Missing values:
name                   0
roaster                0
score                  4
roast_level          405
aroma                126
structure           5085
body                  68
flavour               72
aftertaste           847
coffee_origin        484
roaster_location       4
agtron                 0
cost                2012
agtron_whole         311
dtype: int64


,name,roaster,score,roast_level,aroma,structure,body,flavour,aftertaste,coffee_origin,roaster_location,agtron,cost,agtron_whole
0,Ka’u Classic Dark Roast,Rusty's Hawaiian,95.0,Medium-Dark,9.0,9.0,9.0,9.0,9.0,"Ka‘ū growing district, Hawai'i Island, Hawai'i","Pahala, Hawai'i Island, Hawai'i",44/64,$22.50/7 ounces,44.0
1,Kenya Peaberry Hera,Simon Hsieh Aroma Roast Coffees,95.0,Medium-Dark,9.0,9.0,9.0,9.0,9.0,"Nyeri growing region, south-central Kenya","Taiyuan, Taiwan",44/62,"NT $1,100/227 grams",44.0
2,Ethiopia Washed Limu,Kakalove Cafe,94.0,Medium-Dark,9.0,9.0,9.0,9.0,8.0,"Guji Zone, Oromia Region, southern Ethiopia","Chia-Yi, Taiwan",47/64,NT $290/8 ounces,47.0
3,Strawberry Shadow,modcup,94.0,Medium-Dark,9.0,8.0,9.0,9.0,9.0,"Huila and Quindío Departments, Colombia","Jersey City, New Jersey",46/63,$30.00/250 grams,46.0
4,Ethiopia Uraga Perfume Rose Anaerobic Washed,Linsun Coffee,93.0,Medium-Dark,9.0,8.0,9.0,9.0,8.0,"Guji Zone, Oromia Region, south-central Ethiopia","Tamsui, Taiwan",45/71,NT $800/200 grams,45.0


In [19]:
FEATURES = [
    'roast_level',
    'aroma', 'structure', 'body', 'flavour', 'aftertaste', 'score', 'agtron_whole',
    'roaster', 'coffee_origin', 'roaster_location'
]
TARGET = 'name'

X = df[FEATURES].copy()
y = df[TARGET]

roast_order = [['Light', 'Medium-Light', 'Medium', 'Medium-Dark', 'Dark']]

preprocessor = ColumnTransformer(transformers=[
    ('roast', Pipeline([
        ('encode', OrdinalEncoder(categories=roast_order, handle_unknown='use_encoded_value', unknown_value=-1)),
        ('scale', StandardScaler())
    ]), ['roast_level']),
    ('numeric', Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale', StandardScaler())
    ]), ['aroma', 'structure', 'body', 'flavour', 'aftertaste', 'score', 'agtron_whole']),
    ('categorical', Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]), ['roaster', 'coffee_origin', 'roaster_location'])
])

print('Preprocessor ready.')

Preprocessor ready.


In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'Train: {len(X_train)} samples | Test: {len(X_test)} samples')

Train: 7200 samples | Test: 1800 samples


In [21]:
# sweep k=1 to 20 using cosine metric and distance-weighted voting
# weights=distance means closer neighbours count more — fixes the equal-vote 100%/0% collapse
k_scores = {}
for k in range(1, 21):
    pipe = Pipeline([
        ('prep', preprocessor),
        ('knn', KNeighborsClassifier(n_neighbors=k, metric='cosine', weights='distance'))
    ])
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='accuracy')
    k_scores[k] = scores.mean()

best_k = max(k_scores, key=k_scores.get)
print(f'Best k={best_k}  (CV accuracy: {k_scores[best_k]:.3f})')
for k, score in k_scores.items():
    bar = '#' * int(score * 20)
    print(f'  k={k:2d}: {score:.3f} {bar}')

Best k=13  (CV accuracy: 0.026)
  k= 1: 0.022 
  k= 2: 0.022 
  k= 3: 0.022 
  k= 4: 0.023 
  k= 5: 0.024 
  k= 6: 0.025 
  k= 7: 0.025 
  k= 8: 0.025 
  k= 9: 0.025 
  k=10: 0.025 
  k=11: 0.025 
  k=12: 0.026 
  k=13: 0.026 
  k=14: 0.026 
  k=15: 0.026 
  k=16: 0.025 
  k=17: 0.025 
  k=18: 0.026 
  k=19: 0.026 
  k=20: 0.026 


In [22]:
model = Pipeline([
    ('prep', preprocessor),
    ('knn', KNeighborsClassifier(n_neighbors=best_k, metric='cosine', weights='distance'))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(f'Test accuracy (exact match): {accuracy_score(y_test, y_pred):.3f}')

Test accuracy (exact match): 0.021


In [23]:
# exact match will be low since hundreds of beans share similar scores
# top-N tells us whether the right answer is at least in the shortlist
X_test_transformed = model.named_steps['prep'].transform(X_test)
knn = model.named_steps['knn']
classes = knn.classes_
proba = knn.predict_proba(X_test_transformed)

top3, top5 = 0, 0
for i, true_label in enumerate(y_test.values):
    top_idx = np.argsort(proba[i])[::-1]
    if true_label in classes[top_idx[:3]]: top3 += 1
    if true_label in classes[top_idx[:5]]: top5 += 1

n = len(y_test)
print(f'Top-1 accuracy: {accuracy_score(y_test, y_pred):.3f}')
print(f'Top-3 accuracy: {top3/n:.3f}')
print(f'Top-5 accuracy: {top5/n:.3f}')

Top-1 accuracy: 0.021
Top-3 accuracy: 0.033
Top-5 accuracy: 0.041


In [24]:
def predict_coffee(
        roast_level, aroma, structure, body, flavour, aftertaste,
        score=95, agtron_whole=None,
        roaster=None, coffee_origin=None, roaster_location=None,
        top_n=5):
    input_df = pd.DataFrame([{
        'roast_level':      roast_level,
        'aroma':            aroma,
        'structure':        structure,
        'body':             body,
        'flavour':          flavour,
        'aftertaste':       aftertaste,
        'score':            score,
        'agtron_whole':     agtron_whole,
        'roaster':          roaster,
        'coffee_origin':    coffee_origin,
        'roaster_location': roaster_location
    }])

    X_in = model.named_steps['prep'].transform(input_df)
    p = knn.predict_proba(X_in)[0]
    top_idx = np.argsort(p)[::-1][:top_n]

    print(f'Attributes: roast={roast_level}, aroma={aroma}, structure={structure}, '
          f'body={body}, flavour={flavour}, aftertaste={aftertaste}, score={score}')
    print()
    print(f'Top {top_n} predicted coffees:')
    for rank, idx in enumerate(top_idx, 1):
        print(f'  {rank}. {classes[idx]}  ({p[idx]:.1%})')


# categorical params are optional — imputer fills most_frequent when None
predict_coffee('Light', aroma=9, structure=9, body=9, flavour=10, aftertaste=9)

Attributes: roast=Light, aroma=9, structure=9, body=9, flavour=10, aftertaste=9, score=95

Top 5 predicted coffees:
  1. Kona S12 Kaffa Washed  (9.9%)
  2. Colombia Granja La Esperanza Geisha  (8.8%)
  3. Ninety Plus Gesha Estates Limited Batch #227  (8.5%)
  4. Panama La Esmeralda Geisha Bosque Natural  (7.5%)
  5. Flor Blanca Geisha Colombia  (7.3%)
